In [ ]:
language = 'pt'

## 1. Instalação das Bibliotecas Necessárias

In [ ]:
!pip install git+https://github.com/openai/whisper.git -q
!pip install google-generativeai -q
!pip install gTTS -q
print("Todas as bibliotecas necessárias foram instaladas.")

## 2. Configuração do Ambiente e Imports

In [ ]:
import whisper
import google.generativeai as genai
from gtts import gTTS
from IPython.display import Audio, display, Javascript, clear_output
from google.colab import output, userdata
from base64 import b64decode
import ipywidgets as widgets
import traceback # Para logging de erros mais detalhado


# Configuração da API do Gemini
# Certifique-se de ter sua GOOGLE_API_KEY configurada nos segredos do Colab.
try:
  GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
  genai.configure(api_key=GOOGLE_API_KEY)
  print("API do Gemini configurada com sucesso.")
except Exception as e:
  print(f"Erro ao configurar a API do Gemini: {e}. Certifique-se de que a chave GOOGLE_API_KEY esteja configurada corretamente nos segredos do Colab.")

# Variáveis globais para modelos e chat
model = None # Modelo Whisper será carregado dinamicamente
gemini_model = None # Modelo Gemini será carregado dinamicamente
chat = None # Objeto de chat do Gemini
record_file = None # Caminho do arquivo de áudio gravado
transcription = "" # Texto transcrito
gemini_response = "" # Resposta do Gemini

print("Importações e configurações iniciais concluídas.")

## 3. Elementos da Interface do Usuário (UI) e Funções Auxiliares

In [ ]:
# Botões de Gravação e Reprodução
record_button = widgets.Button(
    description='Iniciar Gravação',
    disabled=False,
    button_style='success',
    tooltip='Clique para iniciar a gravação',
    icon='microphone'
)

stop_button = widgets.Button(
    description='Parar Gravação',
    disabled=True,
    button_style='danger',
    tooltip='Clique para parar a gravação',
    icon='stop'
)

play_button = widgets.Button(
    description='Reproduzir Áudio Gravado',
    disabled=True,
    button_style='info',
    tooltip='Clique para reproduzir o áudio gravado',
    icon='play'
)

# Widget de Saída para Feedback
feedback_output = widgets.Output()

# Código JavaScript para Gravação de Áudio no Navegador
RECORD_JS = """
const sleep  = time => new Promise(resolve => setTimeout(resolve, time))
const b2text = blob => new Promise(resolve => {
  const reader = new FileReader()
  reader.onloadend = e => resolve(e.srcElement.result)
  reader.readAsDataURL(blob)
  })

var stream;
var recorder;
var chunks;

var start_record = () => new Promise(async resolve => {
  stream = await navigator.mediaDevices.getUserMedia({ audio: true})
  recorder = new MediaRecorder(stream)
  chunks = []
  recorder.ondataavailable = e => chunks.push(e.data)
  recorder.start()
  console.log('Recording started...')
})

var stop_record = () => new Promise(async resolve => {
  console.log('Stop recording called, recorder state:', recorder ? recorder.state : 'undefined');
  if (recorder && recorder.state === 'recording') {
    recorder.onstop = async () => { // Assign onstop handler FIRST
      blob = new Blob(chunks)
      text = await b2text(blob)
      resolve(text)
    }
    recorder.stop() // Then call stop()
    stream.getTracks().forEach(track => track.stop()); // Stop microphone access
    console.log('Recording stopped.')
  } else {
    resolve(null); // Retorna null se o gravador não estava ativo
  }
})
"""

display(Javascript(RECORD_JS)) # Injeta o JavaScript globalmente uma única vez
print("Botões de UI e script de gravação JS definidos.")

## 4. Seletores de Modelo e Idioma

In [ ]:
# Dropdown para o Modelo Whisper
whisper_model_selector = widgets.Dropdown(
    options=['tiny', 'base', 'small', 'medium', 'large'],
    value='small',
    description='Modelo Whisper:',
    disabled=False,
)

# Dropdown para o Modelo Gemini
available_gemini_models = []
if 'genai' in globals(): # Verifica se genai foi importado e configurado
    for m in genai.list_models():
        if "generateContent" in m.supported_generation_methods:
            available_gemini_models.append(m.name)

default_gemini_model = 'gemini-1.0-pro'
if default_gemini_model not in available_gemini_models and len(available_gemini_models) > 0:
    default_gemini_model = available_gemini_models[0]
elif len(available_gemini_models) == 0:
    default_gemini_model = None

gemini_model_selector = widgets.Dropdown(
    options=available_gemini_models,
    value=default_gemini_model,
    description='Modelo Gemini:',
    disabled=False,
)

# Dropdown para Seleção de Idioma
language_selector = widgets.Dropdown(
    options={'Português': 'pt', 'English': 'en', 'Español': 'es', 'Français': 'fr', 'Deutsch': 'de'},
    value='pt',
    description='Idioma:',
    disabled=False,
)

print("Seletores de modelo e idioma inicializados.")

## 5. Funções de Callback e Carregamento Inicial de Modelos

In [ ]:
# Funções de Callback para mudança de seleção
def on_whisper_model_change(change):
  global model
  if change['new']:
    with feedback_output:
      clear_output()
      print(f"Carregando modelo Whisper: {change['new']}...")
    try:
      model = whisper.load_model(change['new'])
      with feedback_output:
        print(f"Modelo Whisper '{change['new']}' carregado com sucesso.")
    except Exception as e:
      with feedback_output:
        print(f"Erro ao carregar modelo Whisper '{change['new']}': {e}")

def on_gemini_model_change(change):
  global gemini_model, chat
  if change['new']:
    with feedback_output:
      clear_output()
      print(f"Inicializando modelo Gemini: {change['new']}...")
    try:
      gemini_model = genai.GenerativeModel(change['new'])
      chat = gemini_model.start_chat(history=[]) # Inicia um novo chat para manter o contexto
      with feedback_output:
        print(f"Modelo Gemini '{change['new']}' inicializado com sucesso. Histórico de chat reiniciado.")
    except Exception as e:
      with feedback_output:
        print(f"Erro ao inicializar modelo Gemini '{change['new']}': {e}")

def on_language_change(change):
  global language
  if change['new']:
    language = change['new']
    with feedback_output:
      clear_output()
      print(f"Idioma atualizado para: {language}")

# Carregamento inicial dos modelos com base nos valores padrão dos seletores
with feedback_output:
  clear_output()
  print(f"Carregando modelo Whisper inicial: {whisper_model_selector.value}...")
try:
  model = whisper.load_model(whisper_model_selector.value)
  with feedback_output:
    print(f"Modelo Whisper '{whisper_model_selector.value}' carregado com sucesso.")
except Exception as e:
  with feedback_output:
    print(f"Erro ao carregar modelo Whisper inicial '{whisper_model_selector.value}': {e}")

if gemini_model_selector.value:
    with feedback_output:
      print(f"Inicializando modelo Gemini inicial: {gemini_model_selector.value}...")
    try:
      gemini_model = genai.GenerativeModel(gemini_model_selector.value)
      chat = gemini_model.start_chat(history=[])
      with feedback_output:
        print(f"Modelo Gemini '{gemini_model_selector.value}' inicializado com sucesso.")
    except Exception as e:
      with feedback_output:
        print(f"Erro ao inicializar modelo Gemini inicial '{gemini_model_selector.value}': {e}")
else:
    with feedback_output:
      print("Nenhum modelo Gemini disponível para inicialização. Verifique sua chave API e modelos suportados.")

# Anexar observadores aos dropdowns
whisper_model_selector.observe(on_whisper_model_change, names='value')
gemini_model_selector.observe(on_gemini_model_change, names='value')
language_selector.observe(on_language_change, names='value')

print("Modelos iniciais carregados e observadores configurados.")

## 6. Função Principal de Processamento (`process_and_respond`)

In [ ]:
def process_and_respond(audio_file_path):
  global transcription, gemini_response

  # Re-habilitar botões no início de cada ciclo para garantir que não fiquem desabilitados em caso de erro.
  record_button.disabled = True # Desabilitar enquanto processa
  stop_button.disabled = True
  play_button.disabled = True

  try:
    with feedback_output:
      clear_output()
      print("Transcrevendo áudio... Isso pode levar alguns segundos.")

    # Transcrever o arquivo de áudio
    if model is None:
        raise ValueError("Modelo Whisper não está carregado. Por favor, selecione um modelo.")
    transcription_result = model.transcribe(audio_file_path, fp16=False, language=language)
    transcription = transcription_result["text"]

    with feedback_output:
      print(f"Você disse: {transcription}")
      print("Gerando resposta com Gemini...")

    # Enviar transcrição para o modelo de chat do Gemini para manter o contexto
    if chat is None:
        raise ValueError("Modelo Gemini não está carregado ou chat não inicializado. Por favor, selecione um modelo Gemini.")
    response = chat.send_message(transcription)
    gemini_response = response.text

    with feedback_output:
      print(f"Gemini respondeu: {gemini_response}")
      print("Sintetizando resposta em voz...")

    # Sintetizar a resposta do Gemini usando gTTS
    gtts_object = gTTS(text=gemini_response, lang=language, slow=False)
    response_audio_file = '/content/gemini_response.wav'
    gtts_object.save(response_audio_file)

    with feedback_output:
      print("Reproduzindo resposta...")
    display(Audio(response_audio_file, autoplay=True))

    with feedback_output:
      print("Pronto para a próxima gravação. Clique em 'Iniciar Gravação'.")

  except Exception as e:
    with feedback_output:
      clear_output()
      print(f"Erro no processamento: {e}")
      print(f"Detalhes do erro: {traceback.format_exc()}") # Mostra o traceback completo
      print("Por favor, tente novamente.")

  finally:
    # Sempre re-habilitar o botão de gravação após o processo (sucesso ou falha)
    record_button.disabled = False
    stop_button.disabled = True
    play_button.disabled = (record_file is None) # Habilita play se houver um arquivo gravado

print("'process_and_respond' function definida com tratamento de erros.")

## 7. Funções de Controle da UI e Exibição Final

In [ ]:
# Funções de controle de gravação e reprodução (conectadas aos botões)
def record_audio_callback(b):
  display(Javascript(RECORD_JS)) # Re-adicionado para garantir que o JS esteja sempre definido
  with feedback_output:
    clear_output()
    print("Ouvindo... Clique em 'Parar Gravação' para finalizar.")
  output.eval_js('start_record()')
  record_button.disabled = True
  stop_button.disabled = False
  play_button.disabled = True # Desabilitar reprodução enquanto grava

def stop_and_save_audio_callback(b):
  global record_file
  with feedback_output:
    clear_output()
    print("Processando gravação...")
  js_result = output.eval_js('stop_record()')
  if js_result:
    audio = b64decode(js_result.split(',')[1])
    file_name = 'request_audio.wav'
    record_file = f'/content/{file_name}'
    with open(record_file, 'wb') as f:
      f.write(audio)
    with feedback_output:
      clear_output()
      print(f"Gravação salva em: {record_file}")
    play_button.disabled = False
    # Chamar a função principal de processamento após salvar o áudio
    process_and_respond(record_file)
  else:
    with feedback_output:
      clear_output()
      print("Nenhuma gravação ativa para parar.")
    record_button.disabled = False # Re-habilitar o botão de gravação
    stop_button.disabled = True

def play_recorded_audio_callback(b):
  if record_file:
    with feedback_output:
      clear_output()
      print("Reproduzindo áudio gravado...")
    display(Audio(record_file, autoplay=True))
    with feedback_output:
      print("Reprodução concluída.")
  else:
    with feedback_output:
      clear_output()
      print("Nenhum áudio gravado para reproduzir.")

# Conectar os callbacks aos botões
record_button.on_click(record_audio_callback)
stop_button.on_click(stop_and_save_audio_callback)
play_button.on_click(play_recorded_audio_callback)

# Layout final da UI
ui = widgets.VBox([
    widgets.HBox([record_button, stop_button, play_button]),
    widgets.HBox([whisper_model_selector, gemini_model_selector, language_selector]),
    feedback_output
])

display(ui)

print("UI interativa carregada. Comece clicando em 'Iniciar Gravação'.")